# N107 · 区间DP与最后一步决策

**目标：** 通过选择区间内最后发生的操作分离子问题。

**先修：** N098, N031, N063。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** dp[l][r]；区间长度顺序；边界哨兵；最后操作；合并可行性。

**配套讲解来源：** [同名逐章意见](../../comment/107_interval_dp.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章处理三类"对一排东西做合并/消除操作"的问题，它们有个共同的形态：操作总是发生在某个连续区间内部，而且一次操作会影响邻居关系。三道代表问题是：

1. **戳气球**：一排气球各有分值，戳破第 i 个得 `左邻居×自己×右邻居` 的分，戳破后左右邻居变成相邻。求最大总分。
2. **合并石头**：每次把连续 k 堆合并成 1 堆，代价是这 k 堆重量之和，求把 n 堆合成 1 堆的最小总代价；如果根本合不成 1 堆就返回 −1。
3. **多边形三角剖分**：凸多边形每个顶点带权，任选不相交的对角线把它切成 n−2 个三角形，每个三角形代价是三个顶点权之积，求最小总代价。

拿戳气球给一个具体到数字的例子（“运行示例”部分用的就是这个）：

- 输入 `nums = [3,1,5,8]`。
- 输出 `167`。
- 为什么是 167？最优戳法是：先戳 1（此时邻居是 3 和 5，得 3×1×5=15）→ 再戳 5（邻居是 3 和 8，得 3×5×8=120）→ 再戳 3（邻居是两端的虚拟 1 和 8，得 1×3×8=24）→ 最后戳 8（得 1×8×1=8）。合计 15+120+24+8=167。你可以试试别的顺序，比如先戳 3，只能得 1×3×1=3，远不如留着它当"倍增器"。

## 2. 基础知识：先读懂这些词

- **区间 DP（区间动态规划）**：子问题不再是一个前缀（"前 i 个"），而是一个区间"从 l 到 r"。`dp[l][r]` 表示"只处理区间内部、边界之外的东西都当固定背景"时的最优值。
- **最后一步决策**：本章题眼。在戳气球里，如果你纠结"第一个戳谁"，戳完之后左右两侧的邻居关系全变了，两侧子问题搅在一起；但如果你枚举"区间里**最后**一个戳破的是谁"，那么在它之前，它左右两侧互不影响（左区间的右边界永远是这个最后气球），两侧就能各自独立求最优，再加起来。合并石头（最后一次合并的分界）和三角剖分（挨着底边的最后一个三角形）是同一个思想的三张面孔。
- **开区间状态（`dp[l][r]` 不含端点）**：戳气球和三角剖分的下标 l、r 是"只当背景、不参与操作"的端点——戳气球里它们是哨兵或早已不在的气球，三角剖分里它们是那条弦的两个顶点。
  把状态理解成闭区间会在套转移公式时差一位，读代码前先把这一点钉死。合并石头的 `f(l,r,piles)` 则是左闭右开 `[l,r)`，两边约定不同，更要小心。
- **区间长度顺序（gap 循环）**：计算 `dp[l][r]` 要用到更短的区间（比如 `dp[l][k]`、`dp[k][r]`），所以代码按"区间长度 gap 从小到大"的顺序填表，保证算大区间时小区间已经算好了。
- **边界哨兵**：戳气球代码在数组两端各补一个 `1`（`a=[1]+nums+[1]`）。这两个 1 永远不被戳破，作用是当"虚拟邻居"，这样"戳到只剩一个气球时邻居是虚拟 1"这种情况不需要单独写分支，公式统一了。
- **前缀和（prefix）**：`prefix[i]` 是前 i 堆石头的总重。想马上知道"区间 [l,r) 的石头总重"，用 `prefix[r]-prefix[l]` 一步算出，不用每次现场加一遍。
- **记忆化（memoization，用 `@cache`）**：递归函数把算过的答案自动存起来，同样的参数第二次再来时直接查表返回，不再重算。石头合并用递归+记忆化写，状态是三元组 `(l,r,piles)`。
- **可行性 / 同余条件**：合并石头每次把 k 堆变 1 堆，堆数减少 k−1。要从 n 堆变 1 堆，总共要减掉 n−1，所以 n−1 必须是 k−1 的整数倍，否则无解（返回 −1）。代码里 `(n-1)%(k-1)` 非零就是这个检查。

## 3. 思路推导：从小例子开始

### 以戳气球 [3,1,5,8] 为主线

- **Step 1：加哨兵。** 变成 `a=[1,3,1,5,8,1]`，n=6。我们要的答案是 `dp[0][5]`：在 0 号和 5 号这两个哨兵"之间"把 1~4 号全部戳破的最大得分。
- **Step 2：定状态。** `dp[l][r]` = "把开区间 (l,r) 内部的气球全部戳光"的最大得分。注意是开区间：l 和 r 本身不戳，它们只是当邻居。
- **Step 3：枚举最后一步。** 设 (l,r) 内最后戳破的是 k 号气球。在戳它之前，(l,k) 内部和 (k,r) 内部早已各自戳光，所以此刻 k 的左邻居就是 a[l]、右邻居就是 a[r]，这一步得分固定为 `a[l]*a[k]*a[r]`。于是转移方程：`dp[l][r] = max(dp[l][k] + dp[k][r] + a[l]*a[k]*a[r])`，k 取 l+1 到 r−1。
- **Step 4：按长度从小到大填。** gap=2 的区间里只有 1 个气球，得分就是 `a[l]*a[k]*a[r]`；gap=3、4 的区间依赖更短的，逐层往上。
- **Step 5：手算几个小区间（对应“运行示例”部分的第二张表）。**
  - 空数组 `[]` → 0（没气球可戳）。
  - `[3]` → 哨兵包夹后戳 3，得 1×3×1 = **3**。
  - `[3,1]` → 先戳 1（此时 1 的邻居是 3 和虚拟 1，得 3×1×1=3），再戳 3（邻居是两个虚拟 1，得 1×3×1=3），合计 **6**；反过来先戳 3 只得 1×3×1=3、再戳 1 得 1×1×1=1，合计 4，所以选 6。
  - `[1,5]` → 先戳 1（邻居是虚拟 1 和 5，得 1×1×5=5），再戳 5（邻居是两个虚拟 1，得 1×5×1=5），合计 **10**；若先戳 5（得 1×5×1=5）再戳 1（得 1×1×1=1）只有 6，所以取 10。你运行“运行示例”部分会看到表里就是这个数，建议你先遮住输出自己算一遍再对答案。
  - `[5,8]` → 先戳 5（邻居是虚拟 1 和 8，得 1×5×8=40），再戳 8（得 1×8×1=8），合计 **48**；先戳 8 只有 1×5×8=40 加 1×5×1=5 共 45，取 48。
- **Step 6：合到全区间。** `dp[0][5]`：最优的"最后戳"是 8 号（此时邻居是两个哨兵 1），它先把 (0,4) 内部也就是 `[3,1,5]` 戳到最优 159，最后一步得 1×8×1=8，合计 167。而 `[3,1,5]` 的 159 又来自"最后戳 3"，层层对应第一节那个顺序 1→5→3→8。

### 副线一：合并石头 [3,2,4,1]、k=2 的手算

前缀和 `prefix=[0,3,5,9,10]`。目标 `f(0,4,1)`（区间 [0,4) 压成 1 堆）：

- 最后一次合并必然是把"左边压成的 1 堆"和"右边压成的 1 堆"合成 1 堆。最优分界 m=2：左区间 `[3,2]` 压成 1 堆代价 5（`prefix[2]-prefix[0]`），右区间 `[4,1]` 压成 1 堆代价 5（`prefix[4]-prefix[2]`）。
- 最后把这两堆（5 和 5）合并，代价是区间总和 `prefix[4]-prefix[0]=10`。
- 总计 5+5+10=**20**。注意"每次合并的代价 = 被合并的石头总重"，所以一条固定方案的总代价等于"每个石头被合并的次数×自身重量"，这也是为什么答案要用前缀和一次性加整段。

### 副线二：三角剖分 `[1,3,1,4]` 手算

- 四边形顶点值 `[1,3,1,4]`，只能切一条对角线，两种切法：
  - 对角线 (0,2)：三角形 (0,1,2) 代价 1×3×1=3，加三角形 (0,2,3) 代价 1×1×4=4，合计 7。
  - 对角线 (1,3)：三角形 (0,1,3) 代价 1×3×4=12，加三角形 (1,2,3) 代价 3×1×4=12，合计 24。
- DP 怎么得到 7：`dp[0][3] = min(dp[0][k]+dp[k][3]+values[0]*values[k]*values[3])`，k=1 时是 0+12+12=24，k=2 时是 3+0+4=7，取 7。k=2 正是"贴着弦 (0,3) 的最后一个三角形的第三个顶点"。

## 4. 核心代码：burst_balloons

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def burst_balloons(nums):
    a = [1] + list(nums) + [1]
    n = len(a)
    dp = [[0] * n for _ in range(n)]
    for gap in range(2, n):
        for l in range(n - gap):
            r = l + gap
            dp[l][r] = max((dp[l][k] + dp[k][r] + a[l] * a[k] * a[r] for k in range(l + 1, r)))
    return dp[0][-1]
```

### 逐段读懂代码

### 1. `burst_balloons`

```python
def burst_balloons(nums):
    a = [1] + list(nums) + [1]
    n = len(a)
    dp = [[0] * n for _ in range(n)]
    for gap in range(2, n):
        for l in range(n - gap):
            r = l + gap
            dp[l][r] = max((dp[l][k] + dp[k][r] + a[l] * a[k] * a[r] for k in range(l + 1, r)))
    return dp[0][-1]
```

第一行加哨兵并建 n×n 的表（Python 里 `[0]*n` 生成一行，循环 n 次生成 n 行，不能用 `[[]]*n` 那种共享行的写法，这里写对了）。双重循环 `gap`→`l` 实现"按区间长度从小到大填表"：gap=2 时区间内只有 1 个气球，gap 递增时用到的更短区间都已填好。转移那一行就是 Step 3 的方程，`max(... for k in ...)` 是生成器写法，对每个候选"最后戳的气球"算一个总分再取最大。最后 `dp[0][-1]` 即 `dp[0][n-1]`：在两个哨兵之间戳光所有真气球。

### 2. `merge_stones`

（对应程序见下方分段实现与完整代码。）

逐段看：

- `if k<2: raise ValueError`：k=1 意味着"把 1 堆合并成 1 堆"，题目无意义且会让后续除零（`k-1=0`），直接报错挡住。
- `if not n: return 0`：空数组没有石头，合并代价 0。
- `if (n-1)%(k-1): return -1`：整条可行性判断。每次合并堆数减 k−1，n 堆要变 1 堆必须恰好减 n−1，不整除就无解。
- `prefix=[0]+list(accumulate(stones))`：`accumulate` 生成前缀和（[3,2,4,1] → [3,5,9,10]），前面补 0 让 `prefix[r]-prefix[l]` 可以直接算任意区间和。
- `f(l,r,piles)` = 把 stones[l:r] 压成 piles 堆的最小代价，`@cache` 做记忆化。四个分支依次是：
  1. `length<piles`（石头数比目标堆数还少，不可能）或 `(length-piles)%(k-1)`（同余条件不满足，永远压不到）都返回正无穷，`float('inf')` 表示"这条路废了"，取 min 时自动被淘汰。
  2. 区间只剩 1 堆：它本身就是 1 堆，目标恰是 1 堆则代价 0，否则压不动返回无穷。
  3. 目标是 1 堆：先花 `f(l,r,k)` 把整段压成 k 堆，再一次合并；那次合并的代价是整段总重 `prefix[r]-prefix[l]`。注意"整段和"只在这一处加，不会重复计费。
  4. 一般情况：枚举分界 m，左边 `f(l,m,1)` 压成 1 堆，右边压成 piles−1 堆。m 的步长是 `k-1`，因为左边长度必须满足"能压成 1 堆"的同余条件，跳过不可能的 m 省时间。`default=float('inf')` 表示枚举集合为空时（生成器一个候选都没有）返回无穷，防止 `min()` 对空序列报错。

### 3. `min_score_triangulation`

```python
def min_score_triangulation(values):
    n = len(values)
    if n < 3:
        return 0
    dp = [[0] * n for _ in range(n)]
    for gap in range(2, n):
        for l in range(n - gap):
            r = l + gap
            dp[l][r] = min((dp[l][k] + dp[k][r] + values[l] * values[k] * values[r] for k in range(l + 1, r)))
    return dp[0][-1]
```

它和 `burst_balloons` 骨架完全相同，只有三处不同：开头 `if n<3: return 0` 挡住顶点不足 3 个退化情况；没有加哨兵（因为顶点 l 和 r 本身就是真实顶点，天然充当三角形的两个角）；`max` 换成 `min`（求最小剖分）。转移含义：多边形上从顶点 l 顺时针到顶点 r 这一段（中间隔 gap 条边），最后被切出来的、贴着弦 (l,r) 的那个三角形第三个顶点是 k，于是这一段被分成 (l,k) 和 (k,r) 两个独立子多边形，加上三角形 `values[l]*values[k]*values[r]` 的代价。答案 `dp[0][n-1]` 覆盖整个多边形。顺手验证最小例子 `[1,3,1]`：dp[0][2] 只有一个 k=1，得 1×3×1=3。

## 5. 分段实现：按顺序运行

**本章接口：** `burst_balloons(nums)`、`merge_stones(stones, k)`、`min_score_triangulation(values)`

### 导入本章使用的库

In [1]:
from functools import cache
from itertools import accumulate

### burst_balloons

In [2]:
def burst_balloons(nums):
    a = [1] + list(nums) + [1]
    n = len(a)
    dp = [[0] * n for _ in range(n)]
    for gap in range(2, n):
        for l in range(n - gap):
            r = l + gap
            dp[l][r] = max((dp[l][k] + dp[k][r] + a[l] * a[k] * a[r] for k in range(l + 1, r)))
    return dp[0][-1]

### merge_stones

In [3]:
def merge_stones(stones, k):
    if k < 2:
        raise ValueError('k must be at least 2')
    n = len(stones)
    if not n:
        return 0
    if (n - 1) % (k - 1):
        return -1
    prefix = [0] + list(accumulate(stones))

    @cache
    def f(l, r, piles):
        length = r - l
        if length < piles or (length - piles) % (k - 1):
            return float('inf')
        if length == 1:
            return 0 if piles == 1 else float('inf')
        if piles == 1:
            return f(l, r, k) + prefix[r] - prefix[l]
        return min((f(l, m, 1) + f(m, r, piles - 1) for m in range(l + 1, r, k - 1)), default=float('inf'))
    return f(0, n, 1)

### min_score_triangulation

In [4]:
def min_score_triangulation(values):
    n = len(values)
    if n < 3:
        return 0
    dp = [[0] * n for _ in range(n)]
    for gap in range(2, n):
        for l in range(n - gap):
            r = l + gap
            dp[l][r] = min((dp[l][k] + dp[k][r] + values[l] * values[k] * values[r] for k in range(l + 1, r)))
    return dp[0][-1]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[3,1,5,8]; rows=[]
for k,x in enumerate(a):
    rows.append((k,x))
show_table(['候选最后戳下标','值'],rows)
print('完整区间最优',burst_balloons(a))
# 可直接检查一、二气球区间的独立答案。
show_table(['小输入','最优得分'],[(s,burst_balloons(s)) for s in [[],[3],[3,1],[1,5],[5,8]]])

候选最后戳下标,值
0,3
1,1
2,5
3,8


完整区间最优 167


小输入,最优得分
[],0
[3],3
"[3, 1]",6
"[1, 5]",10
"[5, 8]",48


## 7. 正确性、适用条件与复杂度

固定最后一步后，两侧操作互不改变边界，因此可以组合各自最优解。石头区间压成p堆必须满足长度与p模k−1同余；从k堆合成1堆才加整个区间和，不能在每次分割时重复加。三角形将凸多边形分成两个独立子多边形。

**代价：** 气球和三角剖分O(n³)时间O(n²)空间。石头记忆化使用(l,r,p)状态，保守O(n³k)时间O(n²k)空间上界；递归深度O(n)，仅用于题目小规模区间。

### 展开理解

**为什么是对的？** 关键论证只有一句：一旦固定了"区间里最后发生的那个操作"，这个操作左右两侧的子区间就互不干扰——左区间怎么折腾都碰不到右区间的边界，反之亦然。所以两侧可以各自取最优解，拼起来再加上最后一步本身的得分/代价，就是"以这个最后操作结尾"的所有方案中的最好成绩；对所有候选的最后操作取最大（或最小），就覆盖了所有方案，一个不漏。每个候选方案都对应状态机里的一条真实路径，非法的方案（比如石头压不到目标堆数）在同余检查那里被无穷大值挡掉，永远不会胜出。戳气球里"选第一个操作"之所以不行，是因为戳完第一个后它两侧的气球变成相邻，两个子问题的边界互相依赖，无法独立求最优——这正是"选最后一步"这个设计要规避的坑。

再提醒一个实现层面的坑：**填表必须按区间长度从小到大**。`dp[l][r]` 要读 `dp[l][k]` 和 `dp[k][r]`，这两个区间的长度都严格小于 r−l。如果按"l 从小到大、r 从小到大"的双层循环随手一写，算 `dp[0][3]` 时可能还没算 `dp[1][3]`（它的 l 更大却更短不了），读到的就是残留的 0，答案悄悄错掉。gap 外层循环把这种事故从机制上堵死了。拿第三节的 `[3,1,5]` 试：dp[0][3] 依赖 dp[0][1]、dp[1][3]，其中 dp[1][3]（区间 `[1,5]`，长度 2）必须先于 dp[0][3]（长度 3）算好。

**复杂度**：气球和三角剖分的表有 O(n²) 个格子，每格枚举 O(n) 个 k，所以是 O(n³) 时间、O(n²) 空间。n=100 时大约一百万次操作，瞬间完成；n=500 时约 1.25 亿次，也在可接受范围。石头记忆化的状态是 (l, r, piles) 三元组，piles 最多 k 种，粗略上界是 O(n³k) 时间、O(n²k) 空间，加上递归深度 O(n)——n 很大时可能顶到 Python 的递归上限，所以这套实现适合题目规模较小的区间。

## 8. 单元测试：每个用例在检查什么

- `assert burst_balloons([3,1,5,8])==167`：正常值测试，就是第一节手算的例子，检查"留高分气球当邻居、最后才戳"的核心策略能不能被 DP 找到。
- `assert merge_stones([3,2,4,1],2)==20 and merge_stones([3,2,4,1],3)==-1`：一条断言测两个面。前半段 k=2 是正常值（第三节手算的 20）；后半段 k=3 是可行性测试：n=4、k=3 时 (n−1)%(k−1)=3%2=1 不整除，无论怎么合并都剩不到 1 堆，必须返回 −1。
- `assert min_score_triangulation([1,3,1,4,1,5])==13`：正常值测试，六边形的最优剖分是 LeetCode 1039 的标准例子。你可以手验一种方案：切出 (1,3,1)、(1,1,4)、(1,4,1)、(1,1,5)……总之最小值是 13，若你的实现漏了某个三角形代价项就会偏小或偏大。
- 后半部分是与暴力解的对拍：`pop_ref` 直接模拟戳气球（每次真删一个元素、按当前邻居算分、递归所有选择），`merge_ref` 直接模拟合并（每次真把 k 堆加起来、替换成一个新堆、递归所有位置）。`product(range(1,4),repeat=5)` 枚举所有长度 5、元素取 1/2/3 的 243 种输入，断言三个实现与暴力完全一致。暴力是"照题意直译"的参照物，和它一致就说明 DP 的抽象没有走样。

两个暴力函数里有两处值得驻足的写法。`pop_ref` 里 `(a[i-1] if i else 1)` 的意思是：戳下标 i 的气球时，左邻居是 `a[i-1]`，但 i=0（已经是第一个）时左邻居按虚拟的 1 算，右邻居同理——它用条件表达式把"边界没有邻居"就地解决了。`merge_ref` 里 `a[:i]+(sum(a[i:i+k]),)+a[i+k:]` 用元组拼接真实地"把 k 堆换成一堆"，每一次递归都拿着切好的新数组往下走，没有任何记忆化，所以只适合长度 5 这种玩具规模——这正体现对拍的分工：暴力负责"显然正确"，DP 负责"高效"。

In [6]:
assert burst_balloons([3, 1, 5, 8]) == 167

assert merge_stones([3, 2, 4, 1], 2) == 20 and merge_stones([3, 2, 4, 1], 3) == -1

assert min_score_triangulation([1, 3, 1, 4, 1, 5]) == 13

@cache
def pop_ref(a):
    return max((a[i] * (a[i - 1] if i else 1) * (a[i + 1] if i + 1 < len(a) else 1) + pop_ref(a[:i] + a[i + 1:]) for i in range(len(a))), default=0)

@cache
def merge_ref(a, k):
    if len(a) == 1:
        return 0
    return min((sum(a[i:i + k]) + merge_ref(a[:i] + (sum(a[i:i + k]),) + a[i + k:], k) for i in range(len(a) - k + 1)), default=float('inf'))

from itertools import product

for a in product(range(1, 4), repeat=5):
    assert burst_balloons(a) == pop_ref(a)
    for k in (2, 3):
        assert merge_stones(a, k) == merge_ref(a, k)

print('N107: 所有本章断言通过')

N107: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 解释先选第一个气球难以分离子问题。

**练习 2：** 推导合并到一堆的余数条件。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（解释先选第一个气球为什么难以分离子问题）**：提示——拿 `[3,1,5]` 做手算实验。如果你先戳 1，那么 3 和 5 变成邻居；先戳 3，则 1 的左邻居变成虚拟 1。也就是说"第一步选谁"决定了两侧子问题各自的边界值，左边的答案依赖右边怎么戳，你没法先独立算完左边再算右边。把它和"选最后一步"对比：最后一步的得分 `a[l]*a[k]*a[r]` 里的 l 和 r 是固定不变的边界。写清楚这一点，再用 `pop_ref` 那个暴力函数在小数组上验证你的结论即可。
- **练习 2（推导合并到一堆的余数条件）**：提示——从"每次合并堆数恰好减少 k−1"出发。设从 n 堆做了 t 次合并后剩 1 堆，那么 n − t(k−1) = 1，即 t = (n−1)/(k−1)，t 必须是非负整数，所以 (n−1) 必须被 (k−1) 整除。用一个无解例子手算验证（比如 n=4、k=3：做一次合并剩 2 堆，2<k 合不了，卡死），再说明代码里 `(n-1)%(k-1): return -1` 和 `f` 内部 `(length-piles)%(k-1)` 两处检查分别守全局和局部。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
from functools import cache

from itertools import accumulate

def burst_balloons(nums):
    a = [1] + list(nums) + [1]
    n = len(a)
    dp = [[0] * n for _ in range(n)]
    for gap in range(2, n):
        for l in range(n - gap):
            r = l + gap
            dp[l][r] = max((dp[l][k] + dp[k][r] + a[l] * a[k] * a[r] for k in range(l + 1, r)))
    return dp[0][-1]

def merge_stones(stones, k):
    if k < 2:
        raise ValueError('k must be at least 2')
    n = len(stones)
    if not n:
        return 0
    if (n - 1) % (k - 1):
        return -1
    prefix = [0] + list(accumulate(stones))

    @cache
    def f(l, r, piles):
        length = r - l
        if length < piles or (length - piles) % (k - 1):
            return float('inf')
        if length == 1:
            return 0 if piles == 1 else float('inf')
        if piles == 1:
            return f(l, r, k) + prefix[r] - prefix[l]
        return min((f(l, m, 1) + f(m, r, piles - 1) for m in range(l + 1, r, k - 1)), default=float('inf'))
    return f(0, n, 1)

def min_score_triangulation(values):
    n = len(values)
    if n < 3:
        return 0
    dp = [[0] * n for _ in range(n)]
    for gap in range(2, n):
        for l in range(n - gap):
            r = l + gap
            dp[l][r] = min((dp[l][k] + dp[k][r] + values[l] * values[k] * values[r] for k in range(l + 1, r)))
    return dp[0][-1]

# 示例与回归测试
assert burst_balloons([3, 1, 5, 8]) == 167

assert merge_stones([3, 2, 4, 1], 2) == 20 and merge_stones([3, 2, 4, 1], 3) == -1

assert min_score_triangulation([1, 3, 1, 4, 1, 5]) == 13

@cache
def pop_ref(a):
    return max((a[i] * (a[i - 1] if i else 1) * (a[i + 1] if i + 1 < len(a) else 1) + pop_ref(a[:i] + a[i + 1:]) for i in range(len(a))), default=0)

@cache
def merge_ref(a, k):
    if len(a) == 1:
        return 0
    return min((sum(a[i:i + k]) + merge_ref(a[:i] + (sum(a[i:i + k]),) + a[i + k:], k) for i in range(len(a) - k + 1)), default=float('inf'))

from itertools import product

for a in product(range(1, 4), repeat=5):
    assert burst_balloons(a) == pop_ref(a)
    for k in (2, 3):
        assert merge_stones(a, k) == merge_ref(a, k)

print('N107: 所有本章断言通过')

N107: 所有本章断言通过


## 11. 代表题与迁移

- **312. Burst Balloons**：练"加哨兵 + 枚举最后戳谁 + 按区间长度填表"这套三连，本章 `burst_balloons` 就是它的标准解法。
- **1000. Minimum Cost to Merge Stones**：练"带第三维状态（目标堆数）的区间 DP + 同余可行性判断 + 前缀和计费"，对应 `merge_stones`。
- **1039. Minimum Score Triangulation of Polygon**：练"识别出三角剖分和戳气球同构（枚举贴边的最后三角形）"，对应 `min_score_triangulation`，体会 max 换 min、哨兵可有可无的差别。

三道题共同的心法：见到"操作会影响邻居关系"或"合并有代价"，先问自己"这个区间里最后发生的是什么"，再把区间长度设成外层循环，答案自然浮出来。

**题面入口：** [1000](https://leetcode.com/problems/minimum-cost-to-merge-stones/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。